In [1]:
# =====================================================
# Cell 1 : Import Libraries
# =====================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import random
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xarray as xr

from sklearn.preprocessing import MinMaxScaler
import joblib

print("All libraries imported successfully.")

All libraries imported successfully.


In [2]:
# =====================================================
# Cell 2 : Random Seed
# =====================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print(f"Random Seed = {SEED}")

Random Seed = 42


In [3]:
# =====================================================
# Cell 3 : Project Paths
# =====================================================

PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "Rainfall"

PROCESSED_DIR = PROJECT_ROOT / "processed_data"

PROCESSED_DIR.mkdir(exist_ok=True)

print("Project Root :", PROJECT_ROOT)
print("Raw Data     :", RAW_DATA_DIR)
print("Processed    :", PROCESSED_DIR)

Project Root : C:\Users\ashut\Documents\Rainfall Prediction
Raw Data     : C:\Users\ashut\Documents\Rainfall Prediction\data\Rainfall
Processed    : C:\Users\ashut\Documents\Rainfall Prediction\processed_data


In [4]:
# =====================================================
# Cell 4 : Find NetCDF Files
# =====================================================

nc_files = sorted(RAW_DATA_DIR.glob("*.nc"))

print(f"Total NetCDF files found : {len(nc_files)}")

for file in nc_files:
    print(file.name)

Total NetCDF files found : 25
RF25_ind2000_rfp25.nc
RF25_ind2001_rfp25.nc
RF25_ind2002_rfp25.nc
RF25_ind2003_rfp25.nc
RF25_ind2004_rfp25.nc
RF25_ind2005_rfp25.nc
RF25_ind2006_rfp25.nc
RF25_ind2007_rfp25.nc
RF25_ind2008_rfp25.nc
RF25_ind2009_rfp25.nc
RF25_ind2010_rfp25.nc
RF25_ind2011_rfp25.nc
RF25_ind2012_rfp25.nc
RF25_ind2013_rfp25.nc
RF25_ind2014_rfp25.nc
RF25_ind2015_rfp25.nc
RF25_ind2016_rfp25.nc
RF25_ind2017_rfp25.nc
RF25_ind2018_rfp25.nc
RF25_ind2019_rfp25.nc
RF25_ind2020_rfp25.nc
RF25_ind2021_rfp25.nc
RF25_ind2022_rfp25.nc
RF25_ind2023_rfp25.nc
RF25_ind2024_rfp25.nc


In [5]:
# =====================================================
# Cell 5 : Inspect First NetCDF File
# =====================================================

sample_ds = xr.open_dataset(nc_files[0])

print(sample_ds)

<xarray.Dataset> Size: 26MB
Dimensions:    (TIME: 366, LATITUDE: 129, LONGITUDE: 135)
Coordinates:
  * TIME       (TIME) datetime64[ns] 3kB 2000-01-01 2000-01-02 ... 2000-12-31
  * LATITUDE   (LATITUDE) float64 1kB 6.5 6.75 7.0 7.25 ... 38.0 38.25 38.5
  * LONGITUDE  (LONGITUDE) float64 1kB 66.5 66.75 67.0 ... 99.5 99.75 100.0
Data variables:
    RAINFALL   (TIME, LATITUDE, LONGITUDE) float32 25MB ...
Attributes:
    history:      FERRET V6.82   20-Feb-26
    Conventions:  CF-1.0


In [6]:
# =====================================================
# Cell 6 : Dataset Information
# =====================================================

print("Variables:")
print(list(sample_ds.data_vars))

print("\nCoordinates:")
print(list(sample_ds.coords))

print("\nDimensions:")
print(sample_ds.dims)

Variables:
['RAINFALL']

Coordinates:
['LONGITUDE', 'LATITUDE', 'TIME']

Dimensions:
FrozenMappingWarningOnValuesAccess({'TIME': 366, 'LATITUDE': 129, 'LONGITUDE': 135})


In [7]:
# =====================================================
# Cell 7 : Inspect Rainfall Variable
# =====================================================

# Automatically pick the first data variable
RAIN_VAR = list(sample_ds.data_vars.keys())[0]

print("Rainfall Variable :", RAIN_VAR)

print(sample_ds[RAIN_VAR])

Rainfall Variable : RAINFALL
<xarray.DataArray 'RAINFALL' (TIME: 366, LATITUDE: 129, LONGITUDE: 135)> Size: 25MB
[6373890 values with dtype=float32]
Coordinates:
  * TIME       (TIME) datetime64[ns] 3kB 2000-01-01 2000-01-02 ... 2000-12-31
  * LATITUDE   (LATITUDE) float64 1kB 6.5 6.75 7.0 7.25 ... 38.0 38.25 38.5
  * LONGITUDE  (LONGITUDE) float64 1kB 66.5 66.75 67.0 ... 99.5 99.75 100.0
Attributes:
    long_name:  Rainfall
    units:      mm
    history:    From ind2000_rfp25.grd


In [8]:
# =====================================================
# Cell 8 : Merge All NetCDF Files
# =====================================================

datasets = []

for file in nc_files:
    ds = xr.open_dataset(file)
    datasets.append(ds)

rainfall_ds = xr.concat(datasets, dim="TIME")

print("Datasets merged successfully!")
print(rainfall_ds)

Datasets merged successfully!
<xarray.Dataset> Size: 636MB
Dimensions:    (TIME: 9132, LATITUDE: 129, LONGITUDE: 135)
Coordinates:
  * TIME       (TIME) datetime64[ns] 73kB 2000-01-01 2000-01-02 ... 2024-12-31
  * LATITUDE   (LATITUDE) float64 1kB 6.5 6.75 7.0 7.25 ... 38.0 38.25 38.5
  * LONGITUDE  (LONGITUDE) float64 1kB 66.5 66.75 67.0 ... 99.5 99.75 100.0
Data variables:
    RAINFALL   (TIME, LATITUDE, LONGITUDE) float32 636MB nan nan nan ... nan nan
Attributes:
    history:      FERRET V6.82   20-Feb-26
    Conventions:  CF-1.0


In [9]:
# =====================================================
# Cell 9 : Check Missing Values
# =====================================================

rain = rainfall_ds["RAINFALL"]

missing_values = int(rain.isnull().sum())

print("Missing Values :", missing_values)

Missing Values : 113702611


In [10]:
# =====================================================
# Cell 10 : Fill Missing Values
# =====================================================

rain = rain.fillna(0)

print("NaNs removed successfully!")

NaNs removed successfully!


In [11]:
# =====================================================
# Cell 11 : Create Clean Dataset
# =====================================================

clean_dataset = xr.Dataset(
    {
        "RAINFALL": rain
    }
)

print("Clean dataset created successfully!")

Clean dataset created successfully!


In [12]:
# =====================================================
# Cell 12 : Convert Dataset to NumPy
# =====================================================

rain_array = rain.values.astype(np.float32)

print("Rainfall Array Shape :", rain_array.shape)

Rainfall Array Shape : (9132, 129, 135)


In [13]:
# =====================================================
# Cell 13 : Normalize Rainfall
# =====================================================

scaler = MinMaxScaler()

original_shape = rain_array.shape

rain_scaled = scaler.fit_transform(
    rain_array.reshape(-1, 1)
).reshape(original_shape)

print("Normalization Completed!")
print("Shape :", rain_scaled.shape)

Normalization Completed!
Shape : (9132, 129, 135)


In [14]:
# =====================================================
# Cell 14 : Save Scaler
# =====================================================

joblib.dump(
    scaler,
    PROCESSED_DIR / "scaler.pkl"
)

print("Scaler Saved Successfully!")

Scaler Saved Successfully!


In [15]:
# =====================================================
# Cell 15 : Dataset Statistics
# =====================================================

print("=" * 40)

print("Minimum :", rain_scaled.min())
print("Maximum :", rain_scaled.max())
print("Mean    :", rain_scaled.mean())
print("Std     :", rain_scaled.std())

print("=" * 40)

Minimum : 0.0
Maximum : 0.99999994
Mean    : 0.000900384
Std     : 0.006139398


In [16]:
# =====================================================
# Cell 16 : Sliding Window
# =====================================================

WINDOW_SIZE = 7

X = []
y = []

for i in range(len(rain_scaled) - WINDOW_SIZE):
    X.append(rain_scaled[i:i + WINDOW_SIZE])
    y.append(rain_scaled[i + WINDOW_SIZE])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print("Sliding Window Completed!")
print("X Shape :", X.shape)
print("y Shape :", y.shape)

Sliding Window Completed!
X Shape : (9125, 7, 129, 135)
y Shape : (9125, 129, 135)


In [17]:
# =====================================================
# Cell 17 : Train / Validation / Test Split
# =====================================================

n = len(X)

train_end = int(0.70 * n)
val_end = int(0.85 * n)

X_train = X[:train_end]
y_train = y[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]

print("Train :", X_train.shape)
print("Validation :", X_val.shape)
print("Test :", X_test.shape)

Train : (6387, 7, 129, 135)
Validation : (1369, 7, 129, 135)
Test : (1369, 7, 129, 135)


In [18]:
# =====================================================
# Cell 18 : Save NumPy Files
# =====================================================

np.save(PROCESSED_DIR / "X_train.npy", X_train)
np.save(PROCESSED_DIR / "y_train.npy", y_train)

np.save(PROCESSED_DIR / "X_val.npy", X_val)
np.save(PROCESSED_DIR / "y_val.npy", y_val)

np.save(PROCESSED_DIR / "X_test.npy", X_test)
np.save(PROCESSED_DIR / "y_test.npy", y_test)

print("All NumPy files saved successfully!")

All NumPy files saved successfully!


In [19]:
# =====================================================
# Cell 19 : Save Metadata
# =====================================================

import json

metadata = {
    "variable": "RAINFALL",
    "window_size": WINDOW_SIZE,
    "grid": [129, 135],
    "train_samples": int(len(X_train)),
    "validation_samples": int(len(X_val)),
    "test_samples": int(len(X_test)),
    "normalization": "MinMaxScaler",
    "seed": 42
}

with open(PROCESSED_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Metadata saved successfully!")

Metadata saved successfully!


In [20]:
# =====================================================
# Cell 20 : Save Clean NetCDF
# =====================================================

import gc

sample_ds.close()

for ds in datasets:
    ds.close()

gc.collect()

save_path = PROCESSED_DIR / "clean_rainfall.nc"

clean_dataset.to_netcdf(save_path)

print("clean_rainfall.nc saved successfully!")

clean_rainfall.nc saved successfully!
